# 🎯 Module 09: Hyperparameter Tuning

> **Stage:** Machine Learning  
> **Level:** Intermediate  
> **Module:** 09 — Hyperparameter Tuning  
> **Date:** 19 January 2026  
> **Previous:** 08 — Model Evaluation  
> **Next:** 10 — Feature Selection

---

Models have two kinds of numbers:

- **Parameters** — *learned* during training (weights, biases)
- **Hyperparameters** — *set by you* before training (`n_estimators`, `max_depth`, `learning_rate`)

**Hyperparameter tuning** systematically finds good hyperparameters.

In this notebook we'll cover:

- Parameters vs hyperparameters
- **GridSearchCV**
- **RandomizedSearchCV**
- **Optuna**
- **Bayesian Optimization**
- Tuning the right way (no test-set peeking)
- **Pipelines** + tuning
- Choosing the **scoring metric**

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Distinguish **parameters** from **hyperparameters**
- Use `GridSearchCV` and `RandomizedSearchCV`
- Explain when to prefer random over grid search
- Use **Optuna** for smarter tuning
- Explain **Bayesian Optimization** at a high level
- Tune inside a **Pipeline** (no leakage)
- Pick the right **scoring metric** for the problem
- Explain why the test set must stay untouched

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score, make_scorer

np.random.seed(42)
print('✅ Setup ready.')

In [ ]:
# Load data and split ONCE (test held out for the entire session)
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Class balance:', np.bincount(y_train).tolist())

---
## 🧠 1. Parameters vs Hyperparameters

### Parameters
**Learned** during training.

For Linear Regression:

$$y = wx + b$$

`w` and `b` are learned.

### Hyperparameters
**Set before training**:

```python
RandomForestClassifier(
    n_estimators=200,
    max_depth=10
)
```

- `n_estimators` → hyperparameter
- `max_depth` → hyperparameter
- Tree splits and weights → **learned parameters**

---
## 🎯 2. Why Tune Hyperparameters?

Defaults don't work for every dataset:

```text
max_depth = 2   →  too simple  →  underfitting
max_depth = 50  →  too complex →  overfitting
```

Goal → find a configuration that **generalizes** well.

---
## 🔍 3. Grid Search

**GridSearchCV** tests **every combination** of a predefined grid.

```python
param_grid = {
    'n_estimators': [100, 200],
    'max_depth':    [5, 10, 20],
}
```

**Combinations tested** (2 × 3 = 6):

```text
100, 5    100, 10    100, 20
200, 5    200, 10    200, 20
```

With `cv=5`, each config is trained **5 times** → 6 × 5 = **30 fits**.

### Pros / Cons

- ✅ Simple and exhaustive
- ❌ Explodes with many hyperparameters

In [ ]:
# GridSearchCV on a Random Forest
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth':    [5, 10, 20],
}

grid = GridSearchCV(
    rf, param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=0,
)

t0 = time.time()
grid.fit(X_train, y_train)
t_grid = time.time() - t0

print(f'Time: {t_grid:.2f}s')
print(f'Best params: {grid.best_params_}')
print(f'Best CV F1 : {grid.best_score_:.4f}')

---
## 🎲 4. Randomized Search

**RandomizedSearchCV** samples a **random subset** of the search space.

Instead of testing every combination, it tries `n_iter` random picks.

### Grid vs Random

| GridSearchCV | RandomizedSearchCV |
| --- | --- |
| Every combination | Random combinations |
| Expensive | Usually faster |
| Small spaces | Large spaces |
| Exhaustive | Sample-based |

> On large search spaces, random search often finds **equally good** results with **far fewer trials**.

In [ ]:
# RandomizedSearchCV on the same space (with a bigger one)
from scipy.stats import randint

param_dist = {
    'n_estimators':      randint(50, 500),
    'max_depth':         randint(3, 30),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf':  randint(1, 10),
    'max_features':      ['sqrt', 'log2', None],
}

rnd = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20, cv=5, scoring='f1', random_state=42, n_jobs=-1,
)

t0 = time.time()
rnd.fit(X_train, y_train)
t_rnd = time.time() - t0

print(f'Time: {t_rnd:.2f}s')
print(f'Best params: {rnd.best_params_}')
print(f'Best CV F1 : {rnd.best_score_:.4f}')

In [ ]:
# Compare: number of fits and best CV score
n_fits_grid = 6 * 5               # configs × cv
n_fits_rnd  = 20 * 5              # n_iter × cv

print(f'{"Method":<20} {"Fits":>6} {"Time (s)":>10} {"Best CV F1":>12}')
print('-' * 52)
print(f'{"GridSearch":<20} {n_fits_grid:>6} {t_grid:>10.2f} {grid.best_score_:>12.4f}')
print(f'{"RandomizedSearch":<20} {n_fits_rnd:>6} {t_rnd:>10.2f} {rnd.best_score_:>12.4f}')

---
## 🧪 5. Optuna

**Optuna** is a modern hyperparameter optimization framework.

Instead of a fixed grid or random sampling, Optuna uses **optimization algorithms** to intelligently propose the next trial.

```python
def objective(trial):
    max_depth     = trial.suggest_int('max_depth', 3, 15)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    ...
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
```

### Why it's useful

- Supports **int/float/categorical** spaces
- Prunes bad trials early (`MedianPruner`)
- Uses smarter search (TPE sampler by default)
- Scales from 10 to 1000+ trials

In [ ]:
# Optuna demo — install with: pip install optuna
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 50, 300),
            'max_depth':         trial.suggest_int('max_depth', 3, 25),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        }
        from sklearn.model_selection import cross_val_score
        clf = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
        scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
        return scores.mean()

    study = optuna.create_study(direction='maximize')
    t0 = time.time()
    study.optimize(objective, n_trials=20, show_progress_bar=False)
    t_optuna = time.time() - t0

    print(f'Optuna time: {t_optuna:.2f}s')
    print(f'Best F1: {study.best_value:.4f}')
    print('Best params:', study.best_params)
except ImportError:
    print('Optuna not installed. Run: pip install optuna')

In [ ]:
# Visualize Optuna optimization history
try:
    trials = study.trials
    values = [t.value if t.value is not None else np.nan for t in trials]
    best_so_far = np.maximum.accumulate(np.nan_to_num(values, nan=-np.inf))

    plt.figure(figsize=(8, 4))
    plt.plot(range(len(values)), values, 'o', alpha=0.5, label='trial F1')
    plt.plot(range(len(best_so_far)), best_so_far, '-', color='red', label='best so far')
    plt.xlabel('Trial'); plt.ylabel('F1 score')
    plt.title('Optuna optimization history')
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
except NameError:
    print('Skip — Optuna not available.')

---
## 🧠 6. Bayesian Optimization

**Bayesian Optimization** chooses the next hyperparameters based on results from previous experiments.

Instead of blind search:

```text
Try → Try → Try → Try
```

it **learns from previous trials**:

```text
Trial 1 → Result
           ↓
      Search model
           ↓
Trial 2 → Better region
           ↓
      Search model
           ↓
Trial 3 → Better region
```

### Why it matters

When each trial is **expensive** (minutes/hours), reaching a good config in **fewer trials** is huge.

> Optuna's default sampler (**TPE**) is a form of Bayesian optimization.

---
## 📋 7. Important Hyperparameters by Model

### Random Forest
```text
n_estimators
max_depth
min_samples_split
min_samples_leaf
max_features
```

### XGBoost
```text
n_estimators
learning_rate
max_depth
subsample
colsample_bytree
```

### SVM
```text
C
kernel
gamma
```

### KNN
```text
n_neighbors
weights
metric
```

### Logistic Regression
```text
C
penalty
solver
```

In [ ]:
# Tuning SVM hyperparameters
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, random_state=42)),
])

svm_grid = {
    'svm__C':      [0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma':  ['scale', 'auto'],
}

svm_search = GridSearchCV(
    svm_pipe, svm_grid, cv=5, scoring='f1', n_jobs=-1,
)
svm_search.fit(X_train, y_train)

print(f'Best SVM params: {svm_search.best_params_}')
print(f'Best CV F1     : {svm_search.best_score_:.4f}')

---
## ✅ 8. Tuning With the Correct Evaluation Strategy

**Don't tune against the test set.**

### Correct workflow

```text
Dataset
   ↓
Train + Test Split
   ↓
Hyperparameter Search (CV on TRAIN only)
   ↓
Best Model
   ↓
Final Test Evaluation (once!)
```

Repeated tuning against the test set causes **overfitting to the test data** — your reported score becomes meaningless.

In [ ]:
# FINAL evaluation on the untouched test set
best_model = grid.best_estimator_
y_test_pred = best_model.predict(X_test)

final_f1 = f1_score(y_test, y_test_pred)
print(f'Final test F1 (grid best): {final_f1:.4f}')
print(f'Best CV F1               : {grid.best_score_:.4f}')
print(f'Generalization gap       : {grid.best_score_ - final_f1:+.4f}')

---
## 🏗️ 9. Pipeline + Hyperparameter Tuning

Preprocessing should be **inside the pipeline** so cross-validation is leak-free.

```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression()),
])

params = {'model__C': [0.01, 0.1, 1, 10]}
```

The prefix **`model__`** means "the hyperparameter `C` belongs to the step named `model`."

> This also prevents **data leakage** during cross-validation — the scaler is re-fit inside every fold.

In [ ]:
# Pipeline + GridSearchCV
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=500, random_state=42)),
])

params = {
    'model__C':       [0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l1', 'l2'],
    'model__solver':  ['liblinear', 'saga'],
}

pipe_search = GridSearchCV(
    pipe, params, cv=5, scoring='f1', n_jobs=-1,
)
pipe_search.fit(X_train, y_train)

print(f'Best pipeline params: {pipe_search.best_params_}')
print(f'Best CV F1          : {pipe_search.best_score_:.4f}')

---
## 🎯 10. Choosing a Scoring Metric

**Don't always optimize accuracy.**

| Problem | Metric |
| --- | --- |
| Balanced classification | Accuracy |
| Imbalanced classification | F1 / PR-AUC |
| Avoid false positives | Precision |
| Avoid false negatives | Recall |
| Regression | RMSE / MAE / R² |

> The metric you optimize should reflect the **business objective**.

In [ ]:
# Same search, different scoring metrics
searches = {}
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    s = GridSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {'n_estimators': [100, 200], 'max_depth': [5, 10]},
        cv=5, scoring=metric, n_jobs=-1,
    )
    s.fit(X_train, y_train)
    searches[metric] = s
    print(f'{metric:<10} | best CV {metric}: {s.best_score_:.4f}  | best params: {s.best_params_}')

---
## 🔬 11. Practical Tuning Workflow

```text
Choose Model
     ↓
Identify Important Hyperparameters
     ↓
Define Search Space
     ↓
Choose Metric
     ↓
Choose Cross-Validation
     ↓
Grid / Random / Optuna
     ↓
Find Best Configuration
     ↓
Train Best Model
     ↓
Evaluate Once on Test Set
```

---
## 📝 12. Practice Checklist

- [ ] Train a Random Forest classifier
- [ ] Tune `n_estimators` and `max_depth` with `GridSearchCV`
- [ ] Repeat with `RandomizedSearchCV`
- [ ] Compare the best scores
- [ ] Tune an XGBoost model
- [ ] Experiment with Optuna
- [ ] Use a Pipeline during tuning
- [ ] Try different scoring metrics
- [ ] Explain why the test set should not be used during search

---
## 🏋️ 13. Hands-On Exercises

### Exercise 1 — Bigger Grid
Expand the Random Forest grid with `min_samples_split`, `min_samples_leaf`, and `max_features`. How many fits now?

### Exercise 2 — Random vs Grid
On the same space, run `RandomizedSearchCV` with `n_iter = 10, 20, 50`. Does it beat grid search?

### Exercise 3 — Optuna Pruning
Add `optuna.pruners.MedianPruner()` and use `trial.report(...)` / `trial.should_prune()`. Does training finish faster?

### Exercise 4 — Pipeline Prefix
Build a `ColumnTransformer` with numeric and categorical steps. Tune a hyperparameter inside each step using the `stepname__param` syntax.

### Exercise 5 — Log-scale Search
For `learning_rate` or `C`, use a log-uniform search space. Compare to a linear space.

### Exercise 6 — Overfitting to the Test Set
Deliberately tune against the test set repeatedly. Does the reported score get better while actual performance stalls?

### Exercise 7 — Multiple Models
Tune Logistic Regression, Random Forest, and SVM. Which wins on the same dataset? Is the difference meaningful?

---
## 🎤 14. Interview Questions

1. **What is hyperparameter tuning?**  
   Systematically searching for good hyperparameter values using validation data.

2. **Parameters vs hyperparameters?**  
   Parameters are learned during training (weights); hyperparameters are set before training.

3. **How does GridSearchCV work?**  
   It tests every combination of a predefined grid using cross-validation.

4. **Grid search vs random search?**  
   Grid → exhaustive; random → samples the space. Random is often better for large spaces.

5. **When would you use RandomizedSearchCV?**  
   When the search space is large or training is expensive.

6. **What is Optuna?**  
   A framework for intelligent hyperparameter optimization using sampling + pruning.

7. **What is Bayesian Optimization?**  
   An approach that uses previous trial results to choose the next configuration.

8. **Why use cross-validation during tuning?**  
   To get a more reliable estimate of performance and avoid overfitting to a single split.

9. **Why shouldn't you tune using the test set?**  
   It leaks test information → the reported score overfits the test set and no longer reflects real generalization.

10. **Model selection vs hyperparameter tuning?**  
    Model selection chooses the **algorithm/architecture**; tuning chooses its **hyperparameters**.

11. **How do you choose the scoring metric?**  
    Match it to the cost of errors in the problem — F1 for imbalance, recall for missed positives, etc.

12. **How can hyperparameter tuning cause overfitting?**  
    Repeated tuning against the validation/test set effectively optimizes for that specific split.

---
## 🏁 15. Key Takeaways

- **Hyperparameters** — set before training
- **GridSearchCV** — exhaustive over a defined grid
- **RandomizedSearchCV** — samples a large space efficiently
- **Optuna** — intelligent automated optimization
- **Bayesian Optimization** — uses past trials to pick the next
- Use **cross-validation** during tuning
- Use a **Pipeline** to keep preprocessing leak-free
- Choose the **scoring metric** to match the problem
- **Test set stays untouched** until final evaluation

### 🧭 Golden Rule

> **Tune against validation, evaluate on test, once.** Repeated tuning against the test set is a silent form of overfitting.

---

### 🔗 What's Next?

**Module 10 — Feature Selection**

We'll cover **filter methods**, **wrapper methods (RFE)**, **embedded methods (Lasso, tree importance)**, and **VarianceThreshold** — the tools that help you keep only the features that matter.